# 07 Helix patching (Q2, step 1)

For each operand position and each of the 17 points: the operand's vector is replaced by its helix fit (line + T = 2, 5, 10, 100), by the line or the circles alone, by a 9-dimensional PCA projection, or by the average vector. The fit sees 80 numbers; only prompts with the 20 others are patched (`helixadd.helix.run_patching`). Saved to `results/helix_patching/`. A few minutes on a T4.

Needs `results/number_acts/data.npz` in the repo (notebook 06).

Setup: Accelerator GPU T4, Internet on, secret `HF_TOKEN`.

In [ ]:
!rm -rf helix_addition && git clone https://github.com/amark-23/helix_addition.git
!pip install -q git+https://github.com/safety-research/circuit-tracer@v0.5.3

In [ ]:
import os, sys
os.chdir("/kaggle/working/helix_addition")
sys.path.insert(0, "/kaggle/working/helix_addition/src")
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))

In [ ]:
import numpy as np
from helixadd import model as hmodel, helix
model = hmodel.load_model()
data = dict(np.load("results/number_acts/data.npz"))

In [ ]:
# check: patching every prompt's own vector back in must give the clean logit
tr, te = helix.split()
pairs = helix.patching_pairs("a", te)[:20]
own = data["a"][5][[a for a, b in pairs]]
print(np.abs(helix.patch_logits(model, pairs, 5, helix.A_POS, own) - helix.patch_logits(model, pairs, 5, helix.A_POS, None)).max())

In [ ]:
df = helix.run_patching(model, data)
df

In [ ]:
from IPython.display import FileLink
os.chdir("/kaggle/working")
!zip -qr helix_patching.zip helix_addition/results/helix_patching
FileLink("helix_patching.zip")